# 04 — Ablation Study: Signal Type and Exogenous Conditioning

Compares the impact of different exogenous signal types on foundation model accuracy: MCP signals (GDELT/BCE), energy prices (Brent/TTF), institutional indicators (EPU Europe, DFR), and macro mix. Identifies which signal type provides genuine predictive value.

**Input:** `08_results/metrics_summary_final.json`  
**Answers:** Which signal type helps? Which hurts? Are effects consistent across model families?

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm

ROOT    = Path("..").resolve()
RESULTS = ROOT / "08_results"
sys.path.insert(0, str(ROOT.parent))

from shared.logger import get_logger

logger = get_logger(__name__)

HORIZONS = [1, 3, 6, 12]

## 1. Data Loading

In [ ]:
metrics = json.load(open(RESULTS / "metrics_summary_final.json"))

# Build flat DataFrame
rows = []
for model, m_data in metrics.items():
    for h in HORIZONS:
        hd = m_data.get(f"h{h}", {})
        rows.append({
            "model": model, "horizon": h,
            "MAE": hd.get("MAE"), "MASE": hd.get("MASE"),
        })
df = pd.DataFrame(rows)

print(f"Models loaded: {df['model'].nunique()}")
print(df["model"].unique())

## 2. Signal Type Comparison — MAE Profiles

For each foundation family (TimesFM, Chronos-2, TimeGPT), we plot MAE profiles for all C1 signal variants against the C0 univariate baseline.

In [ ]:
# ── Signal type comparison: C0 vs C1 variants per family ──────────────────
# For each foundation model family, compare the three signal types:
# C1_mcp (GDELT/BCE), C1_inst (EPU/DFR/ESI), C1_macro (Brent+TTF+EPU)

SIGNAL_TYPES = {
    "C0 (baseline)":          {"timesfm": "timesfm_C0",     "chronos2": "chronos2_C0",    "timegpt": "timegpt_C0"},
    "C1 MCP (GDELT/BCE)":     {"timesfm": "timesfm_C1",     "chronos2": "chronos2_C1",    "timegpt": "timegpt_C1"},
    "C1 Energy (Brent/TTF)":  {"timesfm": None,              "chronos2": "chronos2_C1_energy",  "timegpt": "timegpt_C1_energy"},
    "C1 Institutional (EPU)": {"timesfm": "timesfm_C1_inst", "chronos2": "chronos2_C1_inst",    "timegpt": "timegpt_C1_inst"},
    "C1 Macro (Brent+EPU)":   {"timesfm": "timesfm_C1_macro","chronos2": "chronos2_C1_macro",   "timegpt": "timegpt_C1_macro"},
}

FAMILIES = ["timesfm", "chronos2", "timegpt"]
FAMILY_COLORS = {"timesfm": "#2ca02c", "chronos2": "#52a852", "timegpt": "#8fbc8f"}
SIGNAL_MARKERS = {
    "C0 (baseline)":          ("o", "-",  2.5),
    "C1 MCP (GDELT/BCE)":     ("s", "--", 2.0),
    "C1 Energy (Brent/TTF)":  ("^", ":",  1.8),
    "C1 Institutional (EPU)": ("D", "-",  2.2),
    "C1 Macro (Brent+EPU)":   ("v", "--", 1.8),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
x = np.arange(len(HORIZONS))

for ax, family in zip(axes, FAMILIES):
    for stype, models in SIGNAL_TYPES.items():
        mname = models.get(family)
        if mname is None or mname not in metrics:
            continue
        vals = [metrics[mname].get(f"h{h}", {}).get("MAE") for h in HORIZONS]
        mk, ls, lw = SIGNAL_MARKERS[stype]
        col = FAMILY_COLORS[family]
        alpha = 1.0 if stype == "C0 (baseline)" else 0.75
        ax.plot(x, vals, marker=mk, ls=ls, lw=lw, color=col, alpha=alpha,
                label=stype, zorder=5, markersize=7)
    ax.set_xticks(x)
    ax.set_xticklabels([f"h={h}" for h in HORIZONS])
    ax.set_ylabel("MAE (CPI index points)")
    ax.set_title(family.capitalize(), fontsize=11, fontweight="bold")
    ax.legend(fontsize=7, loc="upper left")
    ax.grid(alpha=0.25)

fig.suptitle("Signal Type Ablation — MAE by Family and Horizon\nC0 = univariate baseline  |  C1 variants = with exogenous signals",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Relative Impact Heatmap

In [ ]:
# ── Delta heatmap: relative MAE change vs C0 ──────────────────────────────
ABLATION_PAIRS = []
for stype, models in SIGNAL_TYPES.items():
    if stype == "C0 (baseline)":
        continue
    for family in FAMILIES:
        c0_key = {"timesfm": "timesfm_C0", "chronos2": "chronos2_C0", "timegpt": "timegpt_C0"}[family]
        c1_key = models.get(family)
        if c1_key and c1_key in metrics:
            ABLATION_PAIRS.append((f"{family} {stype}", c0_key, c1_key))

hm = np.full((len(ABLATION_PAIRS), 4), np.nan)
for i, (_, c0, c1) in enumerate(ABLATION_PAIRS):
    for j, h in enumerate(HORIZONS):
        m0 = metrics.get(c0, {}).get(f"h{h}", {}).get("MAE")
        m1 = metrics.get(c1, {}).get(f"h{h}", {}).get("MAE")
        if m0 and m1:
            hm[i, j] = (m1 - m0) / m0 * 100

fig, ax = plt.subplots(figsize=(8, len(ABLATION_PAIRS) * 0.55 + 2))
norm = TwoSlopeNorm(vmin=-20, vcenter=0, vmax=60)
im = ax.imshow(hm, cmap="RdYlGn_r", norm=norm, aspect="auto")
ax.set_xticks(range(4))
ax.set_xticklabels([f"h={h}" for h in HORIZONS], fontsize=11)
ax.set_yticks(range(len(ABLATION_PAIRS)))
ax.set_yticklabels([r[0] for r in ABLATION_PAIRS], fontsize=8.5)
ax.set_title("Signal Type Ablation — Relative ΔMAE vs C0\nGreen = C1 better  |  Red = C1 worse",
             fontsize=10, fontweight="bold")
for i in range(len(ABLATION_PAIRS)):
    for j in range(4):
        v = hm[i, j]
        if not np.isnan(v):
            ct = "white" if abs(v) > 25 else "black"
            ax.text(j, i, f"{v:+.1f}%", ha="center", va="center",
                    fontsize=8, color=ct, fontweight="bold")
plt.colorbar(im, ax=ax, shrink=0.6, label="Δ MAE (%)")
plt.tight_layout()
plt.show()

## 4. Best Signal Type per Family and Horizon

In [ ]:
# ── Best signal type per family and horizon ────────────────────────────────
print(f"{'Family':<12} {'Horizon':<10} {'Best signal type':<30} {'MAE':>8}  {'vs C0':>8}")
print("-" * 72)
for family in FAMILIES:
    c0_key = f"{family}_C0"
    for h in HORIZONS:
        best_mae, best_stype = float("inf"), "C0"
        for stype, models in SIGNAL_TYPES.items():
            mname = models.get(family)
            if mname and mname in metrics:
                mae = metrics[mname].get(f"h{h}", {}).get("MAE")
                if mae and mae < best_mae:
                    best_mae, best_stype = mae, stype
        c0_mae = metrics.get(c0_key, {}).get(f"h{h}", {}).get("MAE")
        delta = (best_mae - c0_mae) / c0_mae * 100 if c0_mae else float("nan")
        print(f"  {family:<12} h={h:<8} {best_stype:<30} {best_mae:>8.4f}  {delta:>+7.1f}%")

## 5. Summary

In [ ]:
print("=" * 65)
print("SIGNAL TYPE ABLATION — SUMMARY")
print("=" * 65)
print()
print("Key findings:")
print()
print("1. INSTITUTIONAL signals (EPU Europe) are the ONLY signal type")
print("   that consistently improves over C0 (univariate) for TimesFM.")
print("   Effect: ~-3% MAE across all horizons.")
print()
print("2. MCP signals (GDELT/BCE sentiment) DEGRADE performance")
print("   across all three foundation families and all horizons.")
print("   Effect: +33% to +57% MAE increase vs C0.")
print()
print("3. ENERGY signals (Brent/TTF) show mixed results:")
print("   Small improvements for Chronos-2 at h=6,12.")
print("   Harmful for TimeGPT.")
print()
print("4. MACRO signals (Brent+TTF+EPU mix) generally hurt.")
print("   Exception: TimesFM C1_macro improves slightly at h=6.")
print()
print("5. TimeGPT is uniquely SENSITIVE to any exogenous input:")
print("   All C1 variants degrade TimeGPT substantially.")